# 05_Statistical_Feature_Analysis

### 1. Imports

In [45]:
import pandas as pd
import numpy as np

from scipy.stats import f_oneway, chi2_contingency

from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

### 2. Load feature-engineered data

In [46]:
data = pd.read_csv(
    'feature_engineered_data.csv'
)

data['HAS_BUREAU_HISTORY'] = (
    data['BUREAU_LOAN_COUNT'].notna()
).astype(int)

data = data.drop(
    columns=['SK_ID_CURR']
)

X = data.drop(columns='TARGET')
y = data['TARGET']

### 3. Reproduce the train/test split

In [47]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(40000, 143)
(10000, 143)


### 4. Remove high-missingness columns

In [48]:
missing_percentage = (
    X_train
    .isnull()
    .mean()
    .mul(100)
)

high_missing_columns = (
    missing_percentage[
        missing_percentage > 60
    ]
    .index
    .tolist()
)

X_train = X_train.drop(
    columns=high_missing_columns
)

X_test = X_test.drop(
    columns=high_missing_columns
)

print("Remaining Features:", X_train.shape[1])

Remaining Features: 126


### 5. Identify numerical and categorical features

In [49]:
numerical_features = (
    X_train
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

categorical_features = (
    X_train
    .select_dtypes(
        include=['object', 'category']
    )
    .columns
    .tolist()
)

print("Numerical:", len(numerical_features))
print("Categorical:", len(categorical_features))

Numerical: 111
Categorical: 15


# Part A: ANOVA for Numerical Features
### 6. Why ANOVA?

For every numerical feature, we test:

H0: Mean feature value is equal for TARGET=0 and TARGET=1.

H1: Mean feature value differs between TARGET=0 and TARGET=1.

However, with 40,000 observations, even a very small difference can produce a tiny p-value.

Therefore, we'll also calculate effect size using η² (eta squared).

Interpretation:

η² ≈ 0.01 → small effect
η² ≈ 0.06 → moderate effect
η² ≈ 0.14 → large effect

These are rough guidelines, not universal thresholds.

### 7. ANOVA function

In [50]:
def anova_analysis(X, y, numerical_features):

    results = []

    for feature in numerical_features:

        group_0 = X.loc[
            y == 0,
            feature
        ].dropna()

        group_1 = X.loc[
            y == 1,
            feature
        ].dropna()

        # Skip unusable features
        if (
            len(group_0) < 2
            or len(group_1) < 2
        ):
            continue

        if (
            group_0.nunique() <= 1
            and group_1.nunique() <= 1
        ):
            continue

        f_statistic, p_value = f_oneway(
            group_0,
            group_1
        )

        combined = pd.concat(
            [group_0, group_1]
        )

        grand_mean = combined.mean()

        ss_between = (
            len(group_0)
            * (group_0.mean() - grand_mean) ** 2
            +
            len(group_1)
            * (group_1.mean() - grand_mean) ** 2
        )

        ss_total = (
            (combined - grand_mean) ** 2
        ).sum()

        eta_squared = (
            ss_between / ss_total
            if ss_total > 0
            else 0
        )

        results.append({
            'FEATURE': feature,
            'F_STATISTIC': f_statistic,
            'P_VALUE': p_value,
            'ETA_SQUARED': eta_squared
        })

    return (
        pd.DataFrame(results)
        .sort_values(
            'ETA_SQUARED',
            ascending=False
        )
        .reset_index(drop=True)
    )

### 8. Run ANOVA

In [51]:
anova_results = anova_analysis(
    X_train,
    y_train,
    numerical_features
)

anova_results.head(20)

,FEATURE,F_STATISTIC,P_VALUE,ETA_SQUARED
0,EXT_SOURCE_3,1086.566156,2.235630e-234,0.032725
1,EXT_SOURCE_2,1013.928235,9.626630e-220,0.024784
2,EXT_SOURCE_1,430.366886,1.862071e-94,0.024067
3,BUREAU_MEAN_DAYS_CREDIT,289.069576,1.458089e-64,0.008346
4,BUREAU_DEBT_CREDIT_RATIO,280.630102,9.767136e-63,0.008173
5,BUREAU_DEBT_LOAN_COUNT,260.292468,2.429416e-58,0.007522
6,BUREAU_ACTIVE_LOAN_COUNT,234.034496,1.173146e-52,0.006768
7,BUREAU_ACTIVE_LOAN_RATIO,227.735413,2.715700e-51,0.006587
8,EMPLOYMENT_YEARS,183.817949,9.214219e-42,0.005571
9,DAYS_EMPLOYED,183.817949,9.214219e-42,0.005571


### 9. Add significance flag

In [52]:
anova_results[
    'STATISTICALLY_SIGNIFICANT'
] = (
    anova_results['P_VALUE'] < 0.05
)

In [53]:
# count
anova_results[
    'STATISTICALLY_SIGNIFICANT'
].value_counts()

STATISTICALLY_SIGNIFICANT
True     69
False    40
Name: count, dtype: int64

### 10. Inspect effect sizes

In [54]:
anova_results[
    [
        'FEATURE',
        'P_VALUE',
        'ETA_SQUARED'
    ]
].head(30)

,FEATURE,P_VALUE,ETA_SQUARED
0,EXT_SOURCE_3,2.235630e-234,0.032725
1,EXT_SOURCE_2,9.626630e-220,0.024784
2,EXT_SOURCE_1,1.862071e-94,0.024067
3,BUREAU_MEAN_DAYS_CREDIT,1.458089e-64,0.008346
4,BUREAU_DEBT_CREDIT_RATIO,9.767136e-63,0.008173
5,BUREAU_DEBT_LOAN_COUNT,2.429416e-58,0.007522
6,BUREAU_ACTIVE_LOAN_COUNT,1.173146e-52,0.006768
7,BUREAU_ACTIVE_LOAN_RATIO,2.715700e-51,0.006587
8,EMPLOYMENT_YEARS,9.214219e-42,0.005571
9,DAYS_EMPLOYED,9.214219e-42,0.005571


In [55]:
# Count features with at least a small effect:
print(
    "Features with η² >= 0.01:",
    (
        anova_results[
            'ETA_SQUARED'
        ] >= 0.01
    ).sum()
)

Features with η² >= 0.01: 3


### 11. Save ANOVA results

In [56]:
anova_results.to_csv(
    'anova_results.csv',
    index=False
)

# Part B: Chi-Square for Categorical Features
### 12. Why Chi-Square?

For every categorical feature:

H0:
Categorical feature and TARGET are independent.

H1:
Categorical feature and TARGET are associated.

Again, we'll calculate both:

p-value and Cramér's V

### 13. Chi-Square function

In [57]:
def chi_square_analysis(
    X,
    y,
    categorical_features
):

    results = []

    for feature in categorical_features:

        feature_data = (
            X[feature]
            .astype('object')
            .fillna('MISSING')
        )

        contingency_table = pd.crosstab(
            feature_data,
            y
        )

        # Need at least 2 categories
        if contingency_table.shape[0] < 2:
            continue

        chi2, p_value, dof, expected = (
            chi2_contingency(
                contingency_table
            )
        )

        n = contingency_table.values.sum()

        r, k = contingency_table.shape

        denominator = min(
            k - 1,
            r - 1
        )

        cramers_v = (
            np.sqrt(
                chi2
                /
                (n * denominator)
            )
            if denominator > 0
            else 0
        )

        results.append({
            'FEATURE': feature,
            'CHI2_STATISTIC': chi2,
            'P_VALUE': p_value,
            'CRAMERS_V': cramers_v
        })

    return (
        pd.DataFrame(results)
        .sort_values(
            'CRAMERS_V',
            ascending=False
        )
        .reset_index(drop=True)
    )

### 14. Run Chi-Square analysis

In [58]:
chi_square_results = chi_square_analysis(
    X_train,
    y_train,
    categorical_features
)

chi_square_results

,FEATURE,CHI2_STATISTIC,P_VALUE,CRAMERS_V
0,OCCUPATION_TYPE,228.501921,1.862181e-38,0.075581
1,ORGANIZATION_TYPE,198.827949,1.363499e-17,0.070503
2,NAME_EDUCATION_TYPE,172.796747,2.625225e-36,0.065726
3,CODE_GENDER,150.084866,1.661155e-34,0.061255
4,NAME_INCOME_TYPE,140.124861,9.432117e-28,0.059187
5,EMERGENCYSTATE_MODE,113.113771,2.739408e-25,0.053177
6,HOUSETYPE_MODE,111.261450,5.872832e-24,0.052740
7,WALLSMATERIAL_MODE,103.526693,2.013785e-19,0.050874
8,NAME_FAMILY_STATUS,78.539985,1.695097e-15,0.044311
9,NAME_CONTRACT_TYPE,67.353787,2.269058e-16,0.041035


### 15. Add significance flag

In [59]:
chi_square_results[
    'STATISTICALLY_SIGNIFICANT'
] = (
    chi_square_results[
        'P_VALUE'
    ] < 0.05
)

In [60]:
# inspect
chi_square_results[
    [
        'FEATURE',
        'P_VALUE',
        'CRAMERS_V',
        'STATISTICALLY_SIGNIFICANT'
    ]
]

,FEATURE,P_VALUE,CRAMERS_V,STATISTICALLY_SIGNIFICANT
0,OCCUPATION_TYPE,1.862181e-38,0.075581,True
1,ORGANIZATION_TYPE,1.363499e-17,0.070503,True
2,NAME_EDUCATION_TYPE,2.625225e-36,0.065726,True
3,CODE_GENDER,1.661155e-34,0.061255,True
4,NAME_INCOME_TYPE,9.432117e-28,0.059187,True
5,EMERGENCYSTATE_MODE,2.739408e-25,0.053177,True
6,HOUSETYPE_MODE,5.872832e-24,0.052740,True
7,WALLSMATERIAL_MODE,2.013785e-19,0.050874,True
8,NAME_FAMILY_STATUS,1.695097e-15,0.044311,True
9,NAME_CONTRACT_TYPE,2.269058e-16,0.041035,True


### 16. Save Chi-Square results

In [61]:
chi_square_results.to_csv(
    'chi_square_results.csv',
    index=False
)

# Part C: Near-Constant Features

Before correlation filtering or VIF, identify numerical features with almost no variation.

### 17. Calculate dominant-value proportion

In [62]:
near_constant_results = []

for feature in numerical_features:

    value_proportions = (
        X_train[feature]
        .value_counts(
            normalize=True,
            dropna=False
        )
    )

    dominant_proportion = (
        value_proportions.iloc[0]
    )

    near_constant_results.append({
        'FEATURE': feature,
        'DOMINANT_PROPORTION':
            dominant_proportion
    })

near_constant_results = (
    pd.DataFrame(
        near_constant_results
    )
    .sort_values(
        'DOMINANT_PROPORTION',
        ascending=False
    )
)

near_constant_results.head(20)

,FEATURE,DOMINANT_PROPORTION
72,FLAG_DOCUMENT_12,1.000000
70,FLAG_DOCUMENT_10,1.000000
10,FLAG_MOBIL,0.999975
62,FLAG_DOCUMENT_2,0.999975
67,FLAG_DOCUMENT_7,0.999850
64,FLAG_DOCUMENT_4,0.999800
77,FLAG_DOCUMENT_17,0.999700
81,FLAG_DOCUMENT_21,0.999625
80,FLAG_DOCUMENT_20,0.999525
79,FLAG_DOCUMENT_19,0.999200


In [63]:
# Flag features where one value accounts for at least 99% of observations:
near_constant_features = (
    near_constant_results.loc[
        near_constant_results[
            'DOMINANT_PROPORTION'
        ] >= 0.99,
        'FEATURE'
    ]
    .tolist()
)

print(
    "Near-constant numerical features:",
    len(near_constant_features)
)

near_constant_features

Near-constant numerical features: 18


['FLAG_DOCUMENT_12',
 'FLAG_DOCUMENT_10',
 'FLAG_MOBIL',
 'FLAG_DOCUMENT_2',
 'FLAG_DOCUMENT_7',
 'FLAG_DOCUMENT_4',
 'FLAG_DOCUMENT_17',
 'FLAG_DOCUMENT_21',
 'FLAG_DOCUMENT_20',
 'FLAG_DOCUMENT_19',
 'FLAG_DOCUMENT_15',
 'FLAG_CONT_MOBILE',
 'FLAG_DOCUMENT_13',
 'FLAG_DOCUMENT_9',
 'FLAG_DOCUMENT_14',
 'FLAG_DOCUMENT_11',
 'FLAG_DOCUMENT_18',
 'FLAG_DOCUMENT_16']

# Part D: Correlation Filtering
### 18. Prepare numerical data
For correlation analysis, median-impute using training medians only.

In [64]:
numerical_for_correlation = (
    X_train[numerical_features]
    .copy()
)

training_medians = (
    numerical_for_correlation
    .median()
)

numerical_for_correlation = (
    numerical_for_correlation
    .fillna(training_medians)
)

In [65]:
# Remove near-constant features:
correlation_features = [
    feature
    for feature in numerical_features
    if feature not in near_constant_features
]

numerical_for_correlation = (
    numerical_for_correlation[
        correlation_features
    ]
)

### 19. Calculate absolute correlation matrix

In [66]:
correlation_matrix = (
    numerical_for_correlation
    .corr()
    .abs()
)

In [67]:
# Use the upper triangle so every pair is examined once.
upper_triangle = (
    correlation_matrix.where(
        np.triu(
            np.ones(
                correlation_matrix.shape
            ),
            k=1
        ).astype(bool)
    )
)

### 20. Find highly correlated pairs
Use:

|correlation| > 0.90

In [68]:
high_correlation_pairs = []

for column in upper_triangle.columns:

    correlated_rows = (
        upper_triangle.index[
            upper_triangle[column] > 0.90
        ]
    )

    for row in correlated_rows:

        high_correlation_pairs.append({
            'FEATURE_1': row,
            'FEATURE_2': column,
            'CORRELATION':
                upper_triangle.loc[
                    row,
                    column
                ]
        })

high_correlation_pairs = pd.DataFrame(
    high_correlation_pairs
)

high_correlation_pairs.sort_values(
    'CORRELATION',
    ascending=False
).head(30)

,FEATURE_1,FEATURE_2,CORRELATION
32,DAYS_BIRTH,AGE,1.000000
33,DAYS_EMPLOYED,EMPLOYMENT_YEARS,1.000000
31,OBS_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,0.998331
17,ELEVATORS_AVG,ELEVATORS_MEDI,0.997256
19,ENTRANCES_AVG,ENTRANCES_MEDI,0.997143
21,FLOORSMAX_AVG,FLOORSMAX_MEDI,0.997078
27,NONLIVINGAREA_AVG,NONLIVINGAREA_MEDI,0.995931
11,APARTMENTS_AVG,APARTMENTS_MEDI,0.995458
25,LIVINGAREA_AVG,LIVINGAREA_MEDI,0.993670
15,YEARS_BEGINEXPLUATATION_AVG,YEARS_BEGINEXPLUATATION_MEDI,0.992886


### 21. Create ANOVA effect-size lookup

In [69]:
eta_lookup = (
    anova_results
    .set_index('FEATURE')[
        'ETA_SQUARED'
    ]
    .to_dict()
)

### 22. Select correlated features for removal

In [70]:
correlation_features_to_drop = set()

for _, row in (
    high_correlation_pairs
    .sort_values(
        'CORRELATION',
        ascending=False
    )
    .iterrows()
):

    feature_1 = row['FEATURE_1']
    feature_2 = row['FEATURE_2']

    # Skip pair if one feature
    # has already been removed
    if (
        feature_1
        in correlation_features_to_drop
        or feature_2
        in correlation_features_to_drop
    ):
        continue

    eta_1 = eta_lookup.get(
        feature_1,
        0
    )

    eta_2 = eta_lookup.get(
        feature_2,
        0
    )

    if eta_1 >= eta_2:
        correlation_features_to_drop.add(
            feature_2
        )
    else:
        correlation_features_to_drop.add(
            feature_1
        )

correlation_features_to_drop = list(
    correlation_features_to_drop
)

print(
    "Features selected for removal due to high correlation:",
    len(correlation_features_to_drop)
)

Features selected for removal due to high correlation: 27


In [71]:
# inspect
correlation_features_to_drop

['AGE',
 'INCOME_PER_PERSON',
 'NONLIVINGAREA_MEDI',
 'ENTRANCES_MODE',
 'NONLIVINGAREA_MODE',
 'ELEVATORS_MEDI',
 'YEARS_BEGINEXPLUATATION_AVG',
 'LANDAREA_MODE',
 'BASEMENTAREA_MEDI',
 'YEARS_BEGINEXPLUATATION_MODE',
 'BUREAU_LOAN_COUNT',
 'FLOORSMAX_MODE',
 'BASEMENTAREA_MODE',
 'REGION_RATING_CLIENT',
 'EMPLOYMENT_YEARS',
 'AMT_CREDIT',
 'LIVINGAREA_AVG',
 'ENTRANCES_MEDI',
 'LANDAREA_AVG',
 'OBS_60_CNT_SOCIAL_CIRCLE',
 'ELEVATORS_MODE',
 'LIVINGAREA_MODE',
 'EMPLOYMENT_AGE_RATIO',
 'FLOORSMAX_MEDI',
 'APARTMENTS_AVG',
 'LIVINGAREA_MEDI',
 'APARTMENTS_MODE']

# Part E: Create Candidate Feature Sets
### 23. Logistic Regression candidate numerical features

In [72]:
significant_numerical_features = (
    anova_results.loc[
        anova_results[
            'P_VALUE'
        ] < 0.05,
        'FEATURE'
    ]
    .tolist()
)

In [73]:
# Remove near-constant and highly correlated features:
logistic_numerical_features = [
    feature
    for feature
    in significant_numerical_features

    if feature
    not in near_constant_features

    and feature
    not in correlation_features_to_drop
]

In [74]:
# Prefer interpretable engineered features while preserving
# one representation of each underlying concept.

if 'DAYS_BIRTH' in logistic_numerical_features:
    logistic_numerical_features.remove('DAYS_BIRTH')

    if 'AGE' not in logistic_numerical_features:
        logistic_numerical_features.append('AGE')


if 'DAYS_EMPLOYED' in logistic_numerical_features:
    logistic_numerical_features.remove('DAYS_EMPLOYED')

    if 'EMPLOYMENT_YEARS' not in logistic_numerical_features:
        logistic_numerical_features.append(
            'EMPLOYMENT_YEARS'
        )

In [75]:
for feature in [
    'AGE',
    'DAYS_BIRTH',
    'EMPLOYMENT_YEARS',
    'DAYS_EMPLOYED'
]:
    print(
        feature,
        "KEPT" if feature in logistic_numerical_features
        else "REMOVED"
    )

AGE KEPT
DAYS_BIRTH REMOVED
EMPLOYMENT_YEARS KEPT
DAYS_EMPLOYED REMOVED


### 24. Logistic Regression candidate categorical features

In [76]:
logistic_categorical_features = (
    chi_square_results.loc[
        chi_square_results[
            'P_VALUE'
        ] < 0.05,
        'FEATURE'
    ]
    .tolist()
)

In [77]:
print(
    "Logistic Numerical Features:",
    len(logistic_numerical_features)
)

print(
    "Logistic Categorical Features:",
    len(logistic_categorical_features)
)

Logistic Numerical Features: 50
Logistic Categorical Features: 12


### 25. Tree-model feature set

In [78]:
tree_numerical_features = [
    feature
    for feature in numerical_features
    if feature not in near_constant_features
]

tree_categorical_features = (
    categorical_features.copy()
)

In [79]:
print(
    "Tree Numerical Features:",
    len(tree_numerical_features)
)

print(
    "Tree Categorical Features:",
    len(tree_categorical_features)
)

Tree Numerical Features: 93
Tree Categorical Features: 15


### 26. Save candidate feature lists

In [80]:
import joblib
import os

os.makedirs(
    'models',
    exist_ok=True
)

joblib.dump(
    logistic_numerical_features,
    'models/logistic_numerical_features.joblib'
)

joblib.dump(
    logistic_categorical_features,
    'models/logistic_categorical_features.joblib'
)

joblib.dump(
    tree_numerical_features,
    'models/tree_numerical_features.joblib'
)

joblib.dump(
    tree_categorical_features,
    'models/tree_categorical_features.joblib'
)

['models/tree_categorical_features.joblib']

# # Statistical Feature Analysis Summary

1. ANOVA was applied to numerical features using training data only to identify differences between defaulters and non-defaulters.

2. Eta-squared effect sizes were calculated alongside p-values because statistical significance alone may overstate the importance of weak effects in large datasets.

3. Chi-Square tests were applied to categorical variables, and Cramér's V was calculated to measure association strength.

4. Near-constant numerical features were identified using dominant-value proportions.

5. Highly correlated numerical feature pairs were detected using an absolute correlation threshold of 0.90.

6. For the Logistic Regression candidate feature set, highly correlated features were filtered by retaining the feature with the larger ANOVA effect size.

7. Separate candidate feature sets were created for linear and tree-based models because multicollinearity and statistical screening affect different model families differently.

8. VIF analysis was deferred until the Logistic Regression candidate set was reduced to a manageable size.

# Part F: VIF Analysis for Logistic Regression
### 27. Prepare numerical data

In [81]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

logistic_vif_data = X_train[
    logistic_numerical_features
].copy()

print(
    "Initial Numerical Features for VIF:",
    logistic_vif_data.shape[1]
)

Initial Numerical Features for VIF: 50


### 28. Median imputation

In [82]:
vif_imputer = SimpleImputer(
    strategy='median'
)

logistic_vif_imputed = (
    vif_imputer.fit_transform(
        logistic_vif_data
    )
)

### 29. Standardize features

In [83]:
vif_scaler = StandardScaler()

logistic_vif_scaled = (
    vif_scaler.fit_transform(
        logistic_vif_imputed
    )
)

In [85]:
# Convert back to DataFrame:
logistic_vif_df = pd.DataFrame(
    logistic_vif_scaled,
    columns=logistic_numerical_features,
    index=X_train.index
) 

### 30. Calculate VIF

In [86]:
from statsmodels.stats.outliers_influence import (
    variance_inflation_factor
)

def calculate_vif(df):

    vif_results = pd.DataFrame({
        'FEATURE': df.columns,

        'VIF': [
            variance_inflation_factor(
                df.values,
                i
            )
            for i in range(
                df.shape[1]
            )
        ]
    })

    return vif_results.sort_values(
        'VIF',
        ascending=False
    ).reset_index(drop=True)

In [87]:
initial_vif_results = calculate_vif(
    logistic_vif_df
)

initial_vif_results.head(20)

,FEATURE,VIF
0,AMT_GOODS_PRICE,10.099938
1,BUREAU_MAX_CREDIT,7.808858
2,REG_CITY_NOT_WORK_CITY,7.162512
3,AMT_ANNUITY,6.838756
4,BUREAU_TOTAL_CREDIT,6.456668
5,LIVE_CITY_NOT_WORK_CITY,5.637945
6,TOTALAREA_MODE,5.397081
7,BUREAU_ACTIVE_LOAN_COUNT,4.936789
8,APARTMENTS_MEDI,4.892208
9,CNT_CHILDREN,4.728326


### 31. Iterative VIF removal

In [89]:
def iterative_vif_selection(
    df,
    threshold=10.0,
    min_features=10
):

    selected_features = (
        df.columns.tolist()
    )

    removal_history = []

    while len(selected_features) > min_features:

        current_data = df[
            selected_features
        ]

        vif_results = calculate_vif(
            current_data
        )

        highest_vif = (
            vif_results.iloc[0]['VIF']
        )

        highest_vif_feature = (
            vif_results.iloc[0]['FEATURE']
        )

        if (
            np.isfinite(highest_vif)
            and highest_vif <= threshold
        ):
            break

        removal_history.append({
            'REMOVED_FEATURE':
                highest_vif_feature,

            'VIF_AT_REMOVAL':
                highest_vif
        })

        selected_features.remove(
            highest_vif_feature
        )

    final_vif_results = calculate_vif(
        df[selected_features]
    )

    removal_history = pd.DataFrame(
        removal_history
    )

    return (
        selected_features,
        final_vif_results,
        removal_history
    )

In [90]:
(
    final_logistic_numerical_features,
    final_vif_results,
    vif_removal_history

) = iterative_vif_selection(

    logistic_vif_df,

    threshold=10.0
)

### 32. Inspect the results

In [91]:
print(
    "Numerical Features Before VIF:",
    len(logistic_numerical_features)
)

print(
    "Numerical Features After VIF:",
    len(final_logistic_numerical_features)
)

print(
    "Features Removed by VIF:",
    len(vif_removal_history)
)

Numerical Features Before VIF: 50
Numerical Features After VIF: 49
Features Removed by VIF: 1


In [92]:
vif_removal_history

,REMOVED_FEATURE,VIF_AT_REMOVAL
0,AMT_GOODS_PRICE,10.099938


In [93]:
final_vif_results

,FEATURE,VIF
0,BUREAU_MAX_CREDIT,7.807729
1,REG_CITY_NOT_WORK_CITY,7.162320
2,BUREAU_TOTAL_CREDIT,6.454681
3,LIVE_CITY_NOT_WORK_CITY,5.637755
4,TOTALAREA_MODE,5.397064
5,BUREAU_ACTIVE_LOAN_COUNT,4.936493
6,APARTMENTS_MEDI,4.892177
7,CNT_CHILDREN,4.723221
8,CNT_FAM_MEMBERS,4.624692
9,BUREAU_DEBT_LOAN_COUNT,4.433490


In [94]:
final_vif_results['VIF'].max()

7.807729369704457

### 33. Final Logistic Regression feature set

In [95]:
final_logistic_categorical_features = (
    logistic_categorical_features.copy()
)

print(
    "Final Logistic Numerical Features:",
    len(final_logistic_numerical_features)
)

print(
    "Final Logistic Categorical Features:",
    len(final_logistic_categorical_features)
)

print(
    "Total Raw Features for Logistic Regression:",
    len(final_logistic_numerical_features)
    +
    len(final_logistic_categorical_features)
)

Final Logistic Numerical Features: 49
Final Logistic Categorical Features: 12
Total Raw Features for Logistic Regression: 61


### 34. Save final feature lists

In [96]:
import joblib

joblib.dump(
    final_logistic_numerical_features,
    'models/final_logistic_numerical_features.joblib'
)

joblib.dump(
    final_logistic_categorical_features,
    'models/final_logistic_categorical_features.joblib'
)

joblib.dump(
    vif_removal_history,
    'models/vif_removal_history.joblib'
)

['models/vif_removal_history.joblib']

### Statistical Feature Analysis — Key Findings

- 69 numerical features showed statistically significant differences between defaulters and non-defaulters at the 5% significance level.
- Only three numerical variables had eta-squared effect sizes above 0.01, demonstrating that most individual predictors had relatively weak effects.
- The three EXT_SOURCE variables exhibited the strongest individual numerical associations with repayment difficulties.
- 12 of 15 categorical features showed statistically significant associations with TARGET.
- OCCUPATION_TYPE, ORGANIZATION_TYPE, and NAME_EDUCATION_TYPE had the largest Cramér's V values, although all categorical association strengths remained below 0.1.
- 18 numerical variables were identified as near-constant and excluded from the candidate feature sets.
- Highly correlated numerical features were filtered for the Logistic Regression candidate set, while a broader feature set was retained for tree-based models.
- Engineered AGE and EMPLOYMENT_YEARS variables were retained instead of their equivalent raw day-based variables for improved interpretability.

### VIF Analysis — Observation

- VIF analysis was performed iteratively on the Logistic Regression numerical candidate features.
- A threshold of 10 was used to identify severe multicollinearity while avoiding overly aggressive feature elimination.
- AMT_GOODS_PRICE was removed because it had the highest VIF (~10.01).
- VIF values were recalculated after each removal because removing one predictor can change the multicollinearity of the remaining variables.
- After iterative filtering, 49 numerical features remained and the maximum VIF decreased to approximately 7.81.
- The final Logistic Regression feature set contained 49 numerical and 12 categorical variables, for a total of 61 raw predictors before one-hot encoding.